# 176 — Aprendizaje continuo y adaptación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Catastrophic forgetting**: al entrenar una red en la tarea B tras dominar la tarea
A, los gradientes de B mueven todos los pesos sin proteger los importantes para A, y
el rendimiento en A colapsa. No es falta de capacidad sino **interferencia**.

Tres familias de solución:

1. **Regularización — EWC** (arXiv:1612.00796): penaliza mover pesos importantes,
   `L = L_B + (λ/2)·Σ F_i (θ_i − θ*_A,i)²`, con `F_i` = Fisher diagonal (importancia).
2. **Replay**: buffer de ejemplos antiguos mezclado en cada minibatch nuevo.
3. **Arquitectura**: módulos nuevos por tarea (Progressive Nets, LoRA por tarea).

Se mide con la matriz `R[i][j]` (rendimiento en tarea j tras entrenar hasta i):
precisión media final, olvido por tarea y backward transfer.


## 🧮 Mecanismo clave: el resorte de EWC

Con dos pesos, `θ*_A = (2, −1)`, `F = (5, 0.1)`, tarea B que quiere `(0, 3)` y `λ=2`:

```text
θ₁: 2θ₁ + 10(θ₁−2) = 0        → θ₁ = 1.67   (F alto → se queda cerca de A)
θ₂: 2(θ₂−3) + 0.2(θ₂+1) = 0   → θ₂ = 2.64   (F bajo → se mueve casi hasta B)
```

La rigidez del resorte es proporcional a la importancia del peso para la tarea vieja.
Con `λ=0` se recupera el fine-tuning ingenuo (olvido total de A).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("frontier", seed=176)
show(result)


## Reflexión

1. En el ejemplo de EWC, ¿qué pasaría con θ₁ si la Fisher del peso 1 fuera 0.1 en
   lugar de 5.0? Calcula el nuevo valor y explica qué significa para la tarea A.
2. El replay guarda datos crudos y EWC solo estadísticos de pesos. ¿En qué escenario
   regulado (salud, banca) esa diferencia decide qué método es viable?
3. ¿Por qué un LLM congelado + RAG no resuelve el dilema estabilidad-plasticidad,
   aunque su conocimiento accesible sí se actualice?
